# Process Awinda (per-joint): IMU IK → ID vs OpenSim ID

Same pipeline as `process_awinda.ipynb`, but runs **single-joint** unilateral TCN checkpoints (hip / knee / ankle) with **zero-phase** input & output filtering.

- **IMU IK**: `mt_processed/{Subject}/ik/VQF/{speed}_{cond}.pkl`
- **OpenSim ID**: `processed/{Subject}/awinda/id/{COND}_{speed}_id.sto`
- **Vicon IK** (sync reference): `processed/{Subject}/awinda/ik/{COND}_{speed}_ik.mot`
- **Sync**: Awinda vs Vicon IK **angle xcorr** (hip+knee, z-scored, 5 s skip) → apply lag to model ID vs OpenSim ID, then **15 s transient trim**
- **Checkpoints** (zero-phase in / zero-phase out):
  - hip: `runs/0512_ik_id_hip_offline_zero_phase/best_model.pt`
  - knee: `runs/0512_ik_id_knee_offline_zero_phase/best_model.pt`
  - ankle: `runs/0512_ik_id_ankle_offline_zero_phase/best_model.pt` *(train if missing)*
- **QC**: optional report of low R² / high RMSE trials — **does not drop trials from the cache**
- **Output**: `analysis/cache/process_awinda_per_joint_{hip,knee,ankle}.npz` — open `visualize_awinda_per_joint.ipynb` / `compare_awinda_full_vs_per_joint.ipynb`

Use kernel `jinwoo-addbiomech` (needs PyTorch).


In [1]:
import json
import pickle
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from scipy.signal import butter, sosfilt, sosfiltfilt, find_peaks

warnings.filterwarnings('ignore', message='.*NumPy.*')

PROJECT_ROOT = Path('/home/metamobility3/Jinwoo/os_kinetics').resolve()
PROCESSED_ROOT = Path('/media/metamobility3/Samsung_T52/Results/processed')
IMU_IK_ROOT = Path('/home/metamobility3/Jinwoo/mt_processed')
IMU_IK_METHOD = 'VQF'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

SUBJECT_MASS_KG = {
    'AB01_Jinwoo': 88.0, 'AB02_Oscar': 71.1, 'AB03_Ilseung': 84.4,
    'AB04_Changseob': 74.0, 'AB05_Maria': 55.0, 'AB06_Jimin': 82.6,
    'AB07_Amy': 51.3, 'AB08_Seokhyun': 71.9,
}

# Full sagittal set used for sync / angle offsets (same as process_awinda.ipynb).
ALL_CHANNELS = [
    'hip_flexion_r', 'knee_angle_r', 'ankle_angle_r',
    'hip_flexion_l', 'knee_angle_l', 'ankle_angle_l',
]

JOINT_SPECS = {
    'hip': {
        'checkpoint': PROJECT_ROOT / 'runs/0512_ik_id_hip_offline_zero_phase/best_model.pt',
        'channels': ['hip_flexion_r', 'hip_flexion_l'],
        'display_names': ['Hip R', 'Hip L'],
    },
    'knee': {
        'checkpoint': PROJECT_ROOT / 'runs/0512_ik_id_knee_offline_zero_phase/best_model.pt',
        'channels': ['knee_angle_r', 'knee_angle_l'],
        'display_names': ['Knee R', 'Knee L'],
    },
    'ankle': {
        'checkpoint': PROJECT_ROOT / 'runs/0512_ik_id_ankle_offline_zero_phase/best_model.pt',
        'channels': ['ankle_angle_r', 'ankle_angle_l'],
        'display_names': ['Ankle R', 'Ankle L'],
    },
}

JOINTS_TO_RUN = ['hip', 'knee', 'ankle']  # edit to subset if needed

DEFAULT_FS_HZ = 100.0
ALIGN_MAX_LAG_SEC = 30.0
XCORR_SKIP_SEC = 5.0
TRANSIENT_TRIM_SEC = 15.0
PREVIEW_FRAMES = 500  # first 5 s at 100 Hz

ANGLE_OFFSET_DEG = {
    ('AB02_Oscar', 'LG_0p8mps'): {c: 6.0 for c in ALL_CHANNELS},
    ('AB01_Jinwoo', 'RD_0p8mps'): {
        'knee_angle_r': 10.0, 'knee_angle_l': 10.0,
        'ankle_angle_r': 10.0, 'ankle_angle_l': 10.0,
    },
}

# Still report under QC thresholds, but do not exclude from FILTERED_TRIAL_DATA.
QC_FORCE_KEEP = {
    'AB02_Oscar::RA_0p8mps',
    'AB08_Seokhyun::RA_0p8mps',
}

sys.path.insert(0, str(PROJECT_ROOT))
from dataset import IK_DOF_NAMES
from model import TCN

ALL_IK_CHANNEL_IDX = [IK_DOF_NAMES.index(c) for c in ALL_CHANNELS]
XCORR_CHANNEL_IDX = [ALL_CHANNELS.index(c) for c in (
    'hip_flexion_r', 'knee_angle_r', 'hip_flexion_l', 'knee_angle_l',
)]

missing = []
available = []
for j in JOINTS_TO_RUN:
    ckpt = JOINT_SPECS[j]['checkpoint']
    if ckpt.is_file():
        available.append(j)
    else:
        missing.append((j, ckpt))

print(f'Device: {DEVICE}')
print(f'Processed root: {PROCESSED_ROOT}')
print(f'Requested joints: {JOINTS_TO_RUN}')
print(f'Available checkpoints: {available}')
if missing:
    print('\nMISSING zero-phase single-joint checkpoint(s) — train before processing:')
    for j, p in missing:
        print(f'  [{j}] {p}')
    print(
        '\nExpected ankle training config (match hip/knee offline zero-phase):\n'
        '  input_mode=sagittal_ankle, output_mode=sagittal_ankle,\n'
        '  input_lowpass_mode=zero_phase, output_lowpass_mode=zero_phase\n'
        '  -> runs/0512_ik_id_ankle_offline_zero_phase/best_model.pt'
    )
if not available:
    raise FileNotFoundError('No per-joint zero-phase checkpoints available to run.')

JOINTS_TO_RUN = available
_offset_print = {f'{s}::{c}': v for (s, c), v in ANGLE_OFFSET_DEG.items()}
print(f'ANGLE_OFFSET_DEG: {_offset_print}')
print(f'QC_FORCE_KEEP: {sorted(QC_FORCE_KEEP)}')


Device: cuda
Processed root: /media/metamobility3/Samsung_T52/Results/processed
Requested joints: ['hip', 'knee', 'ankle']
Available checkpoints: ['hip', 'knee', 'ankle']
ANGLE_OFFSET_DEG: {'AB02_Oscar::LG_0p8mps': {'hip_flexion_r': 6.0, 'knee_angle_r': 6.0, 'ankle_angle_r': 6.0, 'hip_flexion_l': 6.0, 'knee_angle_l': 6.0, 'ankle_angle_l': 6.0}, 'AB01_Jinwoo::RD_0p8mps': {'knee_angle_r': 10.0, 'knee_angle_l': 10.0, 'ankle_angle_r': 10.0, 'ankle_angle_l': 10.0}}
QC_FORCE_KEEP: ['AB02_Oscar::RA_0p8mps', 'AB08_Seokhyun::RA_0p8mps']


In [2]:
def parse_opensim_table(path: Path) -> pd.DataFrame:
    with open(path) as f:
        header_end = next(i for i, line in enumerate(f) if line.strip().lower() == 'endheader')
    return pd.read_csv(path, sep=r'\s+', skiprows=header_end + 1).set_index('time')


def butter_lpf(x, fs_hz, cutoff_hz, order, mode='zero_phase'):
    x = np.asarray(x, dtype=np.float64).reshape(-1)
    nyq = 0.5 * fs_hz
    if cutoff_hz <= 0 or cutoff_hz >= nyq or len(x) < 4:
        return x.copy()
    sos = butter(order, cutoff_hz / nyq, btype='low', output='sos')
    return sosfiltfilt(sos, x) if mode == 'zero_phase' else sosfilt(sos, x)


def lpf_mc(X, fs_hz, cutoff_hz, order, mode='zero_phase'):
    return np.column_stack([butter_lpf(X[:, c], fs_hz, cutoff_hz, order, mode) for c in range(X.shape[1])])


def filename_to_condition(stem: str) -> str:
    speed, cond = stem.split('_', 1)
    return f'{cond.upper()}_{speed}'


def condition_to_pkl_stem(condition: str) -> str:
    cond, speed = condition.split('_', 1)
    return f'{speed}_{cond.lower()}'


def list_available_trials():
    rows = []
    WARNINGS = []
    for subj_dir in sorted(IMU_IK_ROOT.glob('AB*')):
        if not subj_dir.is_dir():
            continue
        subject = subj_dir.name
        if not (PROCESSED_ROOT / subject).is_dir():
            WARNINGS.append(f'[WARN] No processed folder for {subject} — skipping all trials')
            continue
        ik_dir = subj_dir / 'ik' / IMU_IK_METHOD
        if not ik_dir.exists():
            ik_dir = subj_dir
        for pkl_path in sorted(ik_dir.glob('*.pkl')):
            cond = filename_to_condition(pkl_path.stem)
            id_path = PROCESSED_ROOT / subject / 'awinda' / 'id' / f'{cond}_id.sto'
            vicon_path = PROCESSED_ROOT / subject / 'awinda' / 'ik' / f'{cond}_ik.mot'
            ok = id_path.exists() and pkl_path.exists() and vicon_path.exists()
            if not ok:
                WARNINGS.append(f'[WARN] {subject}::{cond} missing id/vicon/pkl — skipped')
            rows.append({
                'subject': subject,
                'condition': cond,
                'pkl_path': pkl_path,
                'id_path': id_path if id_path.exists() else None,
                'vicon_path': vicon_path if vicon_path.exists() else None,
                'mass_kg': SUBJECT_MASS_KG.get(subject, np.nan),
                'ready': ok,
            })
    return pd.DataFrame(rows), WARNINGS


def clip_lag(lag, max_lag_samples):
    clipped = False
    if lag > max_lag_samples:
        lag, clipped = max_lag_samples, True
    elif lag < -max_lag_samples:
        lag, clipped = -max_lag_samples, True
    return int(lag), clipped


def apply_angle_offset_rad(pos_rad: np.ndarray, subject: str, condition: str) -> np.ndarray:
    offsets = ANGLE_OFFSET_DEG.get((subject, condition))
    if not offsets:
        return pos_rad
    out = np.asarray(pos_rad, dtype=np.float64).copy()
    for name, deg in offsets.items():
        out[:, IK_DOF_NAMES.index(name)] += np.deg2rad(float(deg))
    return out


def build_vicon_ik_rad(ik_df: pd.DataFrame) -> np.ndarray:
    pos_deg = np.full((len(ik_df), len(IK_DOF_NAMES)), np.nan)
    for j, name in enumerate(IK_DOF_NAMES):
        if name in ik_df.columns:
            pos_deg[:, j] = ik_df[name].to_numpy(dtype=np.float64)
    return np.deg2rad(pos_deg)


def _zscore_1d(x):
    x = np.asarray(x, dtype=np.float64)
    m = np.isfinite(x)
    if m.sum() < 2:
        return np.zeros_like(x)
    mu, sd = float(np.nanmean(x[m])), float(np.nanstd(x[m]))
    if sd < 1e-9:
        return np.zeros_like(x)
    out = (x - mu) / sd
    out[~m] = 0.0
    return out


def estimate_lag_from_angle_xcorr(
    awinda_rad,
    vicon_rad,
    fs_hz,
    max_lag_samples,
    channel_idx=None,
    skip_s=XCORR_SKIP_SEC,
    angle_cutoff=6.0,
    filter_order=4,
    in_mode='zero_phase',
):
    """Multi-channel z-scored cross-correlation. lag = awinda_idx - vicon_idx."""
    channel_idx = list(channel_idx or XCORR_CHANNEL_IDX)
    awinda_f = lpf_mc(awinda_rad[:, ALL_IK_CHANNEL_IDX], fs_hz, angle_cutoff, filter_order, in_mode)
    vicon_f = lpf_mc(vicon_rad[:, ALL_IK_CHANNEL_IDX], fs_hz, angle_cutoff, filter_order, in_mode)
    skip = int(round(skip_s * fs_hz))
    best_lag, best_score = 0, -np.inf
    for lag in range(-max_lag_samples, max_lag_samples + 1):
        start_a, start_b = max(lag, 0), max(-lag, 0)
        nn = min(len(awinda_f) - start_a, len(vicon_f) - start_b) - skip
        if nn < int(round(2.0 * fs_hz)):
            continue
        seg_a = awinda_f[start_a + skip:start_a + skip + nn]
        seg_b = vicon_f[start_b + skip:start_b + skip + nn]
        score = sum(float(np.dot(_zscore_1d(seg_a[:, c]), _zscore_1d(seg_b[:, c])) / nn) for c in channel_idx)
        if score > best_score:
            best_score, best_lag = score, lag
    lag, clipped = clip_lag(best_lag, max_lag_samples)
    return lag, {'xcorr_score': float(best_score), 'lag_clipped': clipped}


def build_model_input_from_pkl(imu_dict: dict) -> np.ndarray:
    n = len(next(iter(imu_dict.values())))
    pos_deg = np.zeros((n, len(IK_DOF_NAMES)), dtype=np.float64)
    key_map = {
        'hip_flexion_r': 'hip_flexion_r', 'knee_angle_r': 'knee_flexion_r', 'ankle_angle_r': 'ankle_flexion_r',
        'hip_flexion_l': 'hip_flexion_l', 'knee_angle_l': 'knee_flexion_l', 'ankle_angle_l': 'ankle_flexion_l',
    }
    sign_map = {'knee_angle_r': -1.0, 'knee_angle_l': -1.0}
    for ik_name, pkl_name in key_map.items():
        idx = IK_DOF_NAMES.index(ik_name)
        pos_deg[:, idx] = sign_map.get(ik_name, 1.0) * np.asarray(imu_dict[pkl_name], dtype=np.float64)
    return pos_deg


def rmse_r2(y_pred, y_true):
    """RMSE and coefficient of determination R² = 1 - SSE/SST (matches exo analyses)."""
    m = np.isfinite(y_pred) & np.isfinite(y_true)
    if m.sum() < 2:
        return np.nan, np.nan
    e = y_pred[m] - y_true[m]
    rmse = float(np.sqrt(np.mean(e ** 2)))
    ss_res = float(np.sum(e ** 2))
    ss_tot = float(np.sum((y_true[m] - np.mean(y_true[m])) ** 2))
    r2 = float(1.0 - ss_res / (ss_tot + 1e-12))
    return rmse, r2


print('Helpers ready.')


Helpers ready.


In [3]:
def load_joint_model(checkpoint: Path):
    ckpt = torch.load(str(checkpoint), map_location=DEVICE, weights_only=False)
    with open(checkpoint.parent / 'config.json') as f:
        train_cfg = json.load(f)

    cfg = ckpt['model_config']
    model = TCN(**{k: cfg[k] for k in [
        'n_input_channels', 'n_output_channels', 'hidden_channels', 'n_blocks', 'kernel_size', 'dropout',
    ]})
    model.load_state_dict(ckpt['model_state_dict'])
    model.to(DEVICE).eval()

    window_size = int(ckpt.get('window_size', 100))
    input_indices = list(ckpt.get('input_indices'))
    h = len(input_indices) // 2
    input_idx_r, input_idx_l = input_indices[:h], input_indices[h:]

    angle_cutoff = float(train_cfg.get('lowpass_cutoff_hz', 6.0))
    vel_raw = train_cfg.get('velocity_lowpass_cutoff_hz', None)
    # Training falls back to angle cutoff when velocity cutoff is unset.
    vel_cutoff = float(vel_raw) if vel_raw is not None else angle_cutoff
    out_cutoff = float(train_cfg.get('lowpass_cutoff_hz', 6.0))
    filter_order = int(train_cfg.get('lowpass_order', 4))
    in_mode = str(train_cfg.get('input_lowpass_mode', 'zero_phase'))
    out_mode = str(train_cfg.get('output_lowpass_mode', 'zero_phase'))

    if in_mode != 'zero_phase' or out_mode != 'zero_phase':
        raise ValueError(
            f'{checkpoint.parent.name}: expected zero_phase in/out, got in={in_mode!r} out={out_mode!r}'
        )

    @torch.no_grad()
    def infer_one_side(pos_side, vel_side):
        x = np.concatenate([pos_side, vel_side], axis=1).astype(np.float32)
        n, W, c_out = x.shape[0], window_size, cfg['n_output_channels']
        pred = np.zeros((n, c_out), dtype=np.float64)

        def _fwd(start):
            xt = torch.from_numpy(np.ascontiguousarray(x[start:start + W].T)).unsqueeze(0).to(DEVICE)
            return model(xt).squeeze(0).detach().cpu().numpy().T

        pred[:W] = _fwd(0)
        for start in range(1, n - W + 1):
            pred[start + W - 1] = _fwd(start)[W - 1]
        return pred.astype(np.float32)

    def run_bilateral_inference(pos_full, vel_full):
        pr = infer_one_side(pos_full[:, input_idx_r], vel_full[:, input_idx_r])
        pl = infer_one_side(pos_full[:, input_idx_l], vel_full[:, input_idx_l])
        return np.concatenate([pr, pl], axis=1)

    meta = {
        'window_size': window_size,
        'input_indices': input_indices,
        'input_mode': ckpt.get('input_mode'),
        'output_mode': ckpt.get('output_mode'),
        'angle_cutoff': angle_cutoff,
        'vel_cutoff': vel_cutoff,
        'out_cutoff': out_cutoff,
        'filter_order': filter_order,
        'in_mode': in_mode,
        'out_mode': out_mode,
        'n_output_channels': cfg['n_output_channels'],
        'run_bilateral_inference': run_bilateral_inference,
    }
    return meta


JOINT_MODELS = {}
for joint in JOINTS_TO_RUN:
    meta = load_joint_model(JOINT_SPECS[joint]['checkpoint'])
    JOINT_MODELS[joint] = meta
    print(
        f'[{joint}] {JOINT_SPECS[joint]["checkpoint"].parent.name} | '
        f'window={meta["window_size"]} | in_idx={meta["input_indices"]} | '
        f'filters: angle={meta["angle_cutoff"]}Hz/{meta["in_mode"]}, '
        f'vel={meta["vel_cutoff"]}Hz/{meta["in_mode"]}, out={meta["out_cutoff"]}Hz/{meta["out_mode"]}'
    )


[hip] 0512_ik_id_hip_offline_zero_phase | window=100 | in_idx=[6, 13] | filters: angle=6.0Hz/zero_phase, vel=6.0Hz/zero_phase, out=6.0Hz/zero_phase
[knee] 0512_ik_id_knee_offline_zero_phase | window=100 | in_idx=[9, 16] | filters: angle=6.0Hz/zero_phase, vel=6.0Hz/zero_phase, out=6.0Hz/zero_phase
[ankle] 0512_ik_id_ankle_offline_zero_phase | window=100 | in_idx=[10, 17] | filters: angle=6.0Hz/zero_phase, vel=15.0Hz/zero_phase, out=6.0Hz/zero_phase


In [4]:
manifest, WARNINGS = list_available_trials()
for w in WARNINGS:
    print(w)
print(f'\nDiscovered {len(manifest)} IMU trials | ready={int(manifest["ready"].sum())} | skipped={int((~manifest["ready"]).sum())}')
display(manifest[['subject', 'condition', 'ready', 'mass_kg']])



Discovered 40 IMU trials | ready=40 | skipped=0


,subject,condition,ready,mass_kg
0,AB01_Jinwoo,LG_0p8mps,True,88.0
1,AB01_Jinwoo,RA_0p8mps,True,88.0
2,AB01_Jinwoo,RD_0p8mps,True,88.0
3,AB01_Jinwoo,LG_1p2mps,True,88.0
4,AB01_Jinwoo,LG_1p6mps,True,88.0
5,AB02_Oscar,LG_0p8mps,True,71.1
6,AB02_Oscar,RA_0p8mps,True,71.1
7,AB02_Oscar,RD_0p8mps,True,71.1
8,AB02_Oscar,LG_1p2mps,True,71.1
9,AB02_Oscar,LG_1p6mps,True,71.1


In [5]:
JOINT_RESULTS = {}  # joint -> {TRIAL_DATA, summary_df, detail_df, WARNINGS}

for joint in JOINTS_TO_RUN:
    print('\n' + '=' * 72)
    print(f'Processing joint: {joint}')
    print('=' * 72)

    channels = JOINT_SPECS[joint]['channels']
    id_cols = [f'{c}_moment' for c in channels]
    meta = JOINT_MODELS[joint]
    run_bilateral_inference = meta['run_bilateral_inference']
    WINDOW_SIZE = meta['window_size']
    ANGLE_CUTOFF = meta['angle_cutoff']
    VEL_CUTOFF = meta['vel_cutoff']
    OUT_CUTOFF = meta['out_cutoff']
    FILTER_ORDER = meta['filter_order']
    IN_MODE = meta['in_mode']
    OUT_MODE = meta['out_mode']
    n_ch = len(channels)
    sync_ref_idx = 0  # right-side channel of this joint

    TRIAL_DATA = {}
    rows = []
    joint_warnings = []

    for _, row in manifest[manifest['ready']].iterrows():
        subject, cond = row['subject'], row['condition']
        trial_key = f'{subject}::{cond}'
        try:
            imu = pickle.load(open(row['pkl_path'], 'rb'))
            pos_rad = apply_angle_offset_rad(
                np.deg2rad(build_model_input_from_pkl(imu)), subject, cond,
            )
            id_df = parse_opensim_table(row['id_path'])
            t_id = id_df.index.to_numpy(dtype=np.float64)
            fs_hz = 1.0 / float(np.median(np.diff(t_id))) if len(t_id) > 2 else DEFAULT_FS_HZ

            pos_f = lpf_mc(pos_rad, fs_hz, ANGLE_CUTOFF, FILTER_ORDER, IN_MODE)
            vel_f = lpf_mc(np.gradient(pos_f, 1.0 / fs_hz, axis=0), fs_hz, VEL_CUTOFF, FILTER_ORDER, IN_MODE)
            pred_nmpkg_full = run_bilateral_inference(pos_f, vel_f)
            if pred_nmpkg_full.shape[1] != n_ch:
                raise RuntimeError(
                    f'Model outputs {pred_nmpkg_full.shape[1]} channels, expected {n_ch} for {joint}'
                )
            pred_nmpkg_full_f = lpf_mc(pred_nmpkg_full, fs_hz, OUT_CUTOFF, FILTER_ORDER, OUT_MODE)

            id_nm = np.column_stack([
                id_df[c].to_numpy(dtype=np.float64) if c in id_df.columns else np.full(len(t_id), np.nan)
                for c in id_cols
            ])
            id_nmpkg_full_f = lpf_mc(id_nm / row['mass_kg'], fs_hz, OUT_CUTOFF, FILTER_ORDER, OUT_MODE)

            awinda_rad = pos_rad
            vicon_rad = build_vicon_ik_rad(parse_opensim_table(row['vicon_path']))
            if (subject, cond) in ANGLE_OFFSET_DEG:
                print(f'ANGLE OFFSET {trial_key}: {ANGLE_OFFSET_DEG[(subject, cond)]}')
            max_lag = int(round(ALIGN_MAX_LAG_SEC * fs_hz))
            lag, sync_info = estimate_lag_from_angle_xcorr(
                awinda_rad,
                vicon_rad,
                fs_hz,
                max_lag,
                skip_s=XCORR_SKIP_SEC,
                angle_cutoff=ANGLE_CUTOFF,
                filter_order=FILTER_ORDER,
                in_mode=IN_MODE,
            )

            start_pred, start_id = max(lag, 0), max(-lag, 0)
            n_sync = min(len(pred_nmpkg_full_f) - start_pred, len(id_nmpkg_full_f) - start_id)
            trim_n = int(round(TRANSIENT_TRIM_SEC * fs_hz))
            if n_sync - trim_n < WINDOW_SIZE:
                raise RuntimeError(f'Synced window too short after {TRANSIENT_TRIM_SEC}s trim: {n_sync - trim_n}')

            pred_nmpkg_f = pred_nmpkg_full_f[start_pred + trim_n:start_pred + n_sync]
            id_nmpkg_f = id_nmpkg_full_f[start_id + trim_n:start_id + n_sync]
            t_sync = t_id[start_id + trim_n:start_id + n_sync]

            metrics = []
            for c in range(n_ch):
                rmse, r2 = rmse_r2(pred_nmpkg_f[:, c], id_nmpkg_f[:, c])
                metrics.append({'channel': channels[c], 'rmse_nmpkg': rmse, 'r2_nmpkg': r2})

            n_pre = min(PREVIEW_FRAMES, len(pred_nmpkg_full_f), len(id_nmpkg_full_f))
            awinda_hip = np.rad2deg(
                lpf_mc(awinda_rad[:, ALL_IK_CHANNEL_IDX], fs_hz, ANGLE_CUTOFF, FILTER_ORDER, IN_MODE)[:n_pre, 0]
            )
            vicon_hip = np.rad2deg(
                lpf_mc(vicon_rad[:, ALL_IK_CHANNEL_IDX], fs_hz, ANGLE_CUTOFF, FILTER_ORDER, IN_MODE)[:n_pre, 0]
            )
            sync_debug = {
                'fs_hz': fs_hz,
                'sync_method': 'angle_xcorr',
                'xcorr_score': sync_info['xcorr_score'],
                't_id': t_id[:n_pre],
                't_imu': np.arange(n_pre) / fs_hz,
                'pred_pre': pred_nmpkg_full_f[:n_pre],
                'id_pre': id_nmpkg_full_f[:n_pre],
                'pred_ref': pred_nmpkg_full_f[:n_pre, sync_ref_idx],
                'id_ref': id_nmpkg_full_f[:n_pre, sync_ref_idx],
                # keep ankle-named keys for visualize_awinda-style loaders
                'pred_ankle': pred_nmpkg_full_f[:n_pre, sync_ref_idx],
                'id_ankle': id_nmpkg_full_f[:n_pre, sync_ref_idx],
                'awinda_hip_deg': awinda_hip,
                'vicon_hip_deg': vicon_hip,
                'n_sync': n_sync,
                'trim_n': trim_n,
                'start_pred': start_pred,
                'start_id': start_id,
                **sync_info,
            }

            TRIAL_DATA[trial_key] = {
                'subject': subject, 'condition': cond, 't': t_sync, 'mass_kg': row['mass_kg'],
                'pred_nmpkg': pred_nmpkg_f, 'id_nmpkg': id_nmpkg_f, 'metrics': metrics,
                'lag_samples': lag, 'lag_seconds': lag / fs_hz, 'sync_debug': sync_debug,
                'xcorr_score': sync_info['xcorr_score'], 'lag_clipped': sync_info['lag_clipped'],
                'joint': joint,
            }
            mean_rmse = np.nanmean([m['rmse_nmpkg'] for m in metrics])
            mean_r2 = np.nanmean([m['r2_nmpkg'] for m in metrics])
            rows.append({
                'trial': trial_key, 'joint': joint, 'n': len(pred_nmpkg_f),
                'mean_rmse': mean_rmse,
                'mean_r2': mean_r2,
            })
            print(
                f'OK  {trial_key} | lag={lag:+d} samples ({lag / fs_hz:+.3f} s) | '
                f'xcorr={sync_info["xcorr_score"]:.3f} clipped={sync_info["lag_clipped"]} | '
                f'mean RMSE={mean_rmse:.4f} R2={mean_r2:.4f}'
            )
        except Exception as exc:
            joint_warnings.append(f'[WARN] Failed {trial_key}: {exc}')
            print(f'FAIL {trial_key}: {exc}')

    summary_df = pd.DataFrame(rows)
    detail_rows = []
    for trial, d in TRIAL_DATA.items():
        for m in d['metrics']:
            detail_rows.append({'trial': trial, 'joint': joint, **m})
    detail_df = pd.DataFrame(detail_rows)

    print(f'\n[{joint}] Loaded {len(TRIAL_DATA)} trials')
    if not detail_df.empty:
        print(f'\n[{joint}] Per-side mean across trials:')
        display(detail_df.groupby('channel')[['rmse_nmpkg', 'r2_nmpkg']].mean())
        print(f'\n[{joint}] Overall (all trials x sides):')
        print(f"  RMSE = {detail_df['rmse_nmpkg'].mean():.4f} N·m/kg")
        print(f"  R2   = {detail_df['r2_nmpkg'].mean():.4f}")

    JOINT_RESULTS[joint] = {
        'TRIAL_DATA': TRIAL_DATA,
        'summary_df': summary_df,
        'detail_df': detail_df,
        'WARNINGS': joint_warnings,
        'channels': channels,
        'display_names': JOINT_SPECS[joint]['display_names'],
        'checkpoint': str(JOINT_SPECS[joint]['checkpoint']),
    }

print('\nDone. Joints processed:', list(JOINT_RESULTS.keys()))



Processing joint: hip
OK  AB01_Jinwoo::LG_0p8mps | lag=+711 samples (+7.110 s) | xcorr=3.946 clipped=False | mean RMSE=0.1072 R2=0.7242
OK  AB01_Jinwoo::RA_0p8mps | lag=+873 samples (+8.730 s) | xcorr=3.976 clipped=False | mean RMSE=0.1658 R2=0.7265
ANGLE OFFSET AB01_Jinwoo::RD_0p8mps: {'knee_angle_r': 10.0, 'knee_angle_l': 10.0, 'ankle_angle_r': 10.0, 'ankle_angle_l': 10.0}
OK  AB01_Jinwoo::RD_0p8mps | lag=+746 samples (+7.460 s) | xcorr=3.918 clipped=False | mean RMSE=0.1409 R2=0.5735
OK  AB01_Jinwoo::LG_1p2mps | lag=+722 samples (+7.220 s) | xcorr=3.943 clipped=False | mean RMSE=0.1194 R2=0.8563
OK  AB01_Jinwoo::LG_1p6mps | lag=+1300 samples (+13.000 s) | xcorr=3.938 clipped=False | mean RMSE=0.1593 R2=0.8849
ANGLE OFFSET AB02_Oscar::LG_0p8mps: {'hip_flexion_r': 6.0, 'knee_angle_r': 6.0, 'ankle_angle_r': 6.0, 'hip_flexion_l': 6.0, 'knee_angle_l': 6.0, 'ankle_angle_l': 6.0}
OK  AB02_Oscar::LG_0p8mps | lag=+386 samples (+3.860 s) | xcorr=3.947 clipped=False | mean RMSE=0.1177 R2=0.66

,rmse_nmpkg,r2_nmpkg
channel,,
hip_flexion_l,0.175635,0.627401
hip_flexion_r,0.159111,0.678509



[hip] Overall (all trials x sides):
  RMSE = 0.1674 N·m/kg
  R2   = 0.6530

Processing joint: knee
OK  AB01_Jinwoo::LG_0p8mps | lag=+711 samples (+7.110 s) | xcorr=3.946 clipped=False | mean RMSE=0.2380 R2=-1.0808
OK  AB01_Jinwoo::RA_0p8mps | lag=+873 samples (+8.730 s) | xcorr=3.976 clipped=False | mean RMSE=0.1981 R2=0.2550
ANGLE OFFSET AB01_Jinwoo::RD_0p8mps: {'knee_angle_r': 10.0, 'knee_angle_l': 10.0, 'ankle_angle_r': 10.0, 'ankle_angle_l': 10.0}
OK  AB01_Jinwoo::RD_0p8mps | lag=+746 samples (+7.460 s) | xcorr=3.918 clipped=False | mean RMSE=0.5149 R2=-1.3639
OK  AB01_Jinwoo::LG_1p2mps | lag=+722 samples (+7.220 s) | xcorr=3.943 clipped=False | mean RMSE=0.3073 R2=-0.4689
OK  AB01_Jinwoo::LG_1p6mps | lag=+1300 samples (+13.000 s) | xcorr=3.938 clipped=False | mean RMSE=0.3224 R2=0.0399
ANGLE OFFSET AB02_Oscar::LG_0p8mps: {'hip_flexion_r': 6.0, 'knee_angle_r': 6.0, 'ankle_angle_r': 6.0, 'hip_flexion_l': 6.0, 'knee_angle_l': 6.0, 'ankle_angle_l': 6.0}
OK  AB02_Oscar::LG_0p8mps | la

,rmse_nmpkg,r2_nmpkg
channel,,
knee_angle_l,0.184179,0.280964
knee_angle_r,0.155779,0.465508



[knee] Overall (all trials x sides):
  RMSE = 0.1700 N·m/kg
  R2   = 0.3732

Processing joint: ankle
OK  AB01_Jinwoo::LG_0p8mps | lag=+711 samples (+7.110 s) | xcorr=3.946 clipped=False | mean RMSE=0.2463 R2=0.7618
OK  AB01_Jinwoo::RA_0p8mps | lag=+873 samples (+8.730 s) | xcorr=3.976 clipped=False | mean RMSE=0.2970 R2=0.6792
ANGLE OFFSET AB01_Jinwoo::RD_0p8mps: {'knee_angle_r': 10.0, 'knee_angle_l': 10.0, 'ankle_angle_r': 10.0, 'ankle_angle_l': 10.0}
OK  AB01_Jinwoo::RD_0p8mps | lag=+746 samples (+7.460 s) | xcorr=3.918 clipped=False | mean RMSE=0.1423 R2=0.9034
OK  AB01_Jinwoo::LG_1p2mps | lag=+722 samples (+7.220 s) | xcorr=3.943 clipped=False | mean RMSE=0.2889 R2=0.7315
OK  AB01_Jinwoo::LG_1p6mps | lag=+1300 samples (+13.000 s) | xcorr=3.938 clipped=False | mean RMSE=0.2599 R2=0.8113
ANGLE OFFSET AB02_Oscar::LG_0p8mps: {'hip_flexion_r': 6.0, 'knee_angle_r': 6.0, 'ankle_angle_r': 6.0, 'hip_flexion_l': 6.0, 'knee_angle_l': 6.0, 'ankle_angle_l': 6.0}
OK  AB02_Oscar::LG_0p8mps | lag

,rmse_nmpkg,r2_nmpkg
channel,,
ankle_angle_l,0.204464,0.824895
ankle_angle_r,0.222611,0.801714



[ankle] Overall (all trials x sides):
  RMSE = 0.2135 N·m/kg
  R2   = 0.8133

Done. Joints processed: ['hip', 'knee', 'ankle']


In [6]:
CACHE_DIR = PROJECT_ROOT / 'analysis' / 'cache'
CACHE_DIR.mkdir(parents=True, exist_ok=True)


def _trial_key_to_prefix(trial_key: str) -> str:
    return trial_key.replace('::', '__')


def save_processed_awinda_per_joint_cache(joint: str, result: dict, path: Path):
    trial_data = result['TRIAL_DATA']
    summary_df = result['summary_df']
    detail_df = result['detail_df']
    channels = result['channels']
    if not trial_data:
        raise RuntimeError(f'[{joint}] TRIAL_DATA is empty — run processing cell first.')

    payload = {
        'joint': np.array(joint),
        'channels': np.array(channels),
        'display_names': np.array(result['display_names']),
        'checkpoint': np.array(result['checkpoint']),
        'trial_keys': np.array(sorted(trial_data.keys()), dtype=object),
        'sync_method': np.array('angle_xcorr'),
        'xcorr_skip_sec': np.array(XCORR_SKIP_SEC),
        'transient_trim_sec': np.array(TRANSIENT_TRIM_SEC),
        'preview_frames': np.array(PREVIEW_FRAMES),
    }
    if summary_df is not None and not summary_df.empty:
        payload['summary_df'] = np.array(summary_df.to_json(orient='split'), dtype=object)
    if detail_df is not None and not detail_df.empty:
        payload['detail_df'] = np.array(detail_df.to_json(orient='split'), dtype=object)

    for trial_key, d in trial_data.items():
        p = _trial_key_to_prefix(trial_key)
        payload[f'{p}__t'] = np.asarray(d['t'], dtype=np.float64)
        payload[f'{p}__pred_nmpkg'] = np.asarray(d['pred_nmpkg'], dtype=np.float32)
        payload[f'{p}__id_nmpkg'] = np.asarray(d['id_nmpkg'], dtype=np.float32)
        payload[f'{p}__meta'] = np.array([
            d['subject'], d['condition'], d['mass_kg'],
            d['lag_samples'], d['lag_seconds'],
            d['xcorr_score'], d['lag_clipped'],
        ], dtype=object)
        rmse = [m['rmse_nmpkg'] for m in d['metrics']]
        r2 = [m['r2_nmpkg'] for m in d['metrics']]
        payload[f'{p}__rmse_nmpkg'] = np.asarray(rmse, dtype=np.float64)
        payload[f'{p}__r2_nmpkg'] = np.asarray(r2, dtype=np.float64)

        sd = d['sync_debug']
        payload[f'{p}__sync_fs_hz'] = np.array(sd['fs_hz'])
        payload[f'{p}__sync_n'] = np.array(sd['n_sync'])
        payload[f'{p}__sync_t_id'] = np.asarray(sd['t_id'], dtype=np.float64)
        payload[f'{p}__sync_t_imu'] = np.asarray(sd['t_imu'], dtype=np.float64)
        payload[f'{p}__sync_pred_pre'] = np.asarray(sd['pred_pre'], dtype=np.float32)
        payload[f'{p}__sync_id_pre'] = np.asarray(sd['id_pre'], dtype=np.float32)
        payload[f'{p}__sync_pred_ankle'] = np.asarray(sd['pred_ankle'], dtype=np.float32)
        payload[f'{p}__sync_id_ankle'] = np.asarray(sd['id_ankle'], dtype=np.float32)
        if 'awinda_hip_deg' in sd:
            payload[f'{p}__sync_awinda_hip_deg'] = np.asarray(sd['awinda_hip_deg'], dtype=np.float32)
            payload[f'{p}__sync_vicon_hip_deg'] = np.asarray(sd['vicon_hip_deg'], dtype=np.float32)
        if 'xcorr_score' in sd:
            payload[f'{p}__sync_xcorr_score'] = np.array(sd['xcorr_score'])

    np.savez_compressed(str(path), **payload)
    print(f'[{joint}] Saved {len(trial_data)} trials -> {path}')


CACHE_PATHS = {}
for joint, result in JOINT_RESULTS.items():
    path = CACHE_DIR / f'process_awinda_per_joint_{joint}.npz'
    save_processed_awinda_per_joint_cache(joint, result, path)
    CACHE_PATHS[joint] = path

print('\nCaches (all trials):', {j: str(p) for j, p in CACHE_PATHS.items()})


[hip] Saved 40 trials -> /home/metamobility3/Jinwoo/os_kinetics/analysis/cache/process_awinda_per_joint_hip.npz
[knee] Saved 40 trials -> /home/metamobility3/Jinwoo/os_kinetics/analysis/cache/process_awinda_per_joint_knee.npz
[ankle] Saved 40 trials -> /home/metamobility3/Jinwoo/os_kinetics/analysis/cache/process_awinda_per_joint_ankle.npz

Caches (all trials): {'hip': '/home/metamobility3/Jinwoo/os_kinetics/analysis/cache/process_awinda_per_joint_hip.npz', 'knee': '/home/metamobility3/Jinwoo/os_kinetics/analysis/cache/process_awinda_per_joint_knee.npz', 'ankle': '/home/metamobility3/Jinwoo/os_kinetics/analysis/cache/process_awinda_per_joint_ankle.npz'}


In [7]:
MIN_R2 = 0.7
MAX_RMSE = 0.2

FILTERED_JOINT_RESULTS = {}
EXCLUDED_BY_JOINT = {}

for joint, result in JOINT_RESULTS.items():
    summary_df = result['summary_df']
    TRIAL_DATA = result['TRIAL_DATA']
    if summary_df.empty:
        print(f'[{joint}] No processed trials — skip QC.')
        continue

    below = summary_df[
        (summary_df['mean_r2'] < MIN_R2) | (summary_df['mean_rmse'] > MAX_RMSE)
    ].copy()
    force_kept = below[below['trial'].isin(QC_FORCE_KEEP)].copy()
    flagged = below[~below['trial'].isin(QC_FORCE_KEEP)].copy()
    if not flagged.empty:
        flagged[['subject', 'condition']] = flagged['trial'].str.split('::', n=1, expand=True)
        flagged['fail_r2'] = flagged['mean_r2'] < MIN_R2
        flagged['fail_rmse'] = flagged['mean_rmse'] > MAX_RMSE

    kept = summary_df[~summary_df['trial'].isin(flagged['trial'])].copy()

    print(
        f'\n[{joint}] QC report (informational only — cache keeps ALL trials): '
        f'mean R2 < {MIN_R2} or mean RMSE > {MAX_RMSE} N·m/kg\n'
        f'  below threshold: {len(below)} / {len(summary_df)} trials\n'
        f'  force-kept (not flagged): {len(force_kept)}\n'
        f'  flagged: {len(flagged)}\n'
        f'  above threshold: {len(kept)} trials'
    )
    if not force_kept.empty:
        print('  force-kept trials:', ', '.join(sorted(force_kept['trial'])))

    if flagged.empty:
        print('  No trials flagged.')
    else:
        print('  Flagged trials by subject:')
        for subject in sorted(flagged['subject'].unique()):
            sub = flagged[flagged['subject'] == subject].sort_values('condition')
            trials = []
            for _, row in sub.iterrows():
                reasons = []
                if row['fail_r2']:
                    reasons.append(f"R2={row['mean_r2']:.3f}")
                if row['fail_rmse']:
                    reasons.append(f"RMSE={row['mean_rmse']:.3f}")
                trials.append(f"{row['condition']} ({', '.join(reasons)})")
            print(f'    {subject}: {", ".join(trials)}')
        display(
            flagged.sort_values(['subject', 'condition'])[
                ['subject', 'condition', 'mean_r2', 'mean_rmse', 'n', 'fail_r2', 'fail_rmse']
            ].reset_index(drop=True)
        )

    EXCLUDED_BY_JOINT[joint] = set(flagged['trial'])
    FILTERED_JOINT_RESULTS[joint] = {
        **result,
        'TRIAL_DATA': {k: v for k, v in TRIAL_DATA.items() if k not in EXCLUDED_BY_JOINT[joint]},
    }
    print(
        f'  Note: main cache retains all {len(TRIAL_DATA)} trials; '
        f'FILTERED_TRIAL_DATA has {len(FILTERED_JOINT_RESULTS[joint]["TRIAL_DATA"])} '
        f'(for optional viz only).'
    )

print('\nMain caches unchanged (all trials):', {j: str(p) for j, p in CACHE_PATHS.items()})



[hip] QC report (informational only — cache keeps ALL trials): mean R2 < 0.7 or mean RMSE > 0.2 N·m/kg
  below threshold: 21 / 40 trials
  force-kept (not flagged): 1
  flagged: 20
  above threshold: 20 trials
  force-kept trials: AB08_Seokhyun::RA_0p8mps
  Flagged trials by subject:
    AB01_Jinwoo: RD_0p8mps (R2=0.574)
    AB02_Oscar: LG_0p8mps (R2=0.666), LG_1p2mps (R2=0.402, RMSE=0.254), RD_0p8mps (R2=-0.021, RMSE=0.212)
    AB03_Ilseung: LG_0p8mps (R2=0.620), RD_0p8mps (R2=0.481)
    AB04_Changseob: LG_0p8mps (R2=0.673), RA_0p8mps (RMSE=0.212), RD_0p8mps (R2=0.202)
    AB05_Maria: LG_0p8mps (R2=0.234, RMSE=0.312), RD_0p8mps (R2=0.106, RMSE=0.217)
    AB06_Jimin: LG_0p8mps (R2=0.425), RA_0p8mps (R2=0.535, RMSE=0.284), RD_0p8mps (R2=0.426)
    AB07_Amy: LG_0p8mps (R2=0.514), RA_0p8mps (R2=0.650, RMSE=0.242), RD_0p8mps (R2=0.277)
    AB08_Seokhyun: LG_0p8mps (R2=0.690), LG_1p2mps (R2=0.640, RMSE=0.209), RD_0p8mps (R2=0.397, RMSE=0.221)


,subject,condition,mean_r2,mean_rmse,n,fail_r2,fail_rmse
0,AB01_Jinwoo,RD_0p8mps,0.573505,0.140884,6500,True,False
1,AB02_Oscar,LG_0p8mps,0.666363,0.117727,6500,True,False
2,AB02_Oscar,LG_1p2mps,0.402027,0.253702,6500,True,True
3,AB02_Oscar,RD_0p8mps,-0.020715,0.212452,6500,True,True
4,AB03_Ilseung,LG_0p8mps,0.619854,0.111752,5350,True,False
5,AB03_Ilseung,RD_0p8mps,0.481422,0.152955,5248,True,False
6,AB04_Changseob,LG_0p8mps,0.672525,0.128218,6500,True,False
7,AB04_Changseob,RA_0p8mps,0.745778,0.212451,6500,False,True
8,AB04_Changseob,RD_0p8mps,0.202439,0.167314,6500,True,False
9,AB05_Maria,LG_0p8mps,0.234418,0.311914,6500,True,True


  Note: main cache retains all 40 trials; FILTERED_TRIAL_DATA has 20 (for optional viz only).

[knee] QC report (informational only — cache keeps ALL trials): mean R2 < 0.7 or mean RMSE > 0.2 N·m/kg
  below threshold: 22 / 40 trials
  force-kept (not flagged): 2
  flagged: 20
  above threshold: 20 trials
  force-kept trials: AB02_Oscar::RA_0p8mps, AB08_Seokhyun::RA_0p8mps
  Flagged trials by subject:
    AB01_Jinwoo: LG_0p8mps (R2=-1.081, RMSE=0.238), LG_1p2mps (R2=-0.469, RMSE=0.307), LG_1p6mps (R2=0.040, RMSE=0.322), RA_0p8mps (R2=0.255), RD_0p8mps (R2=-1.364, RMSE=0.515)
    AB02_Oscar: LG_0p8mps (R2=-4.281, RMSE=0.263), LG_1p2mps (R2=0.289), LG_1p6mps (R2=0.599, RMSE=0.252), RD_0p8mps (R2=0.537, RMSE=0.277)
    AB04_Changseob: LG_0p8mps (R2=-0.216), LG_1p2mps (R2=0.584), RD_0p8mps (R2=0.627, RMSE=0.200)
    AB05_Maria: LG_0p8mps (R2=-0.017, RMSE=0.383)
    AB06_Jimin: LG_0p8mps (R2=0.519), RA_0p8mps (R2=0.488)
    AB07_Amy: LG_0p8mps (R2=0.524), RA_0p8mps (R2=0.621)
    AB08_Seokhy

,subject,condition,mean_r2,mean_rmse,n,fail_r2,fail_rmse
0,AB01_Jinwoo,LG_0p8mps,-1.080822,0.237951,6500,True,True
1,AB01_Jinwoo,LG_1p2mps,-0.468910,0.307283,6500,True,True
2,AB01_Jinwoo,LG_1p6mps,0.039876,0.322432,6500,True,True
3,AB01_Jinwoo,RA_0p8mps,0.254976,0.198077,6500,True,False
4,AB01_Jinwoo,RD_0p8mps,-1.363886,0.514866,6500,True,True
5,AB02_Oscar,LG_0p8mps,-4.281130,0.262644,6500,True,True
6,AB02_Oscar,LG_1p2mps,0.289281,0.145218,6500,True,False
7,AB02_Oscar,LG_1p6mps,0.599494,0.252297,6500,True,True
8,AB02_Oscar,RD_0p8mps,0.537062,0.276838,6500,True,True
9,AB04_Changseob,LG_0p8mps,-0.215921,0.175038,6500,True,False


  Note: main cache retains all 40 trials; FILTERED_TRIAL_DATA has 20 (for optional viz only).

[ankle] QC report (informational only — cache keeps ALL trials): mean R2 < 0.7 or mean RMSE > 0.2 N·m/kg
  below threshold: 15 / 40 trials
  force-kept (not flagged): 1
  flagged: 14
  above threshold: 26 trials
  force-kept trials: AB08_Seokhyun::RA_0p8mps
  Flagged trials by subject:
    AB01_Jinwoo: LG_0p8mps (RMSE=0.246), LG_1p2mps (RMSE=0.289), LG_1p6mps (RMSE=0.260), RA_0p8mps (R2=0.679, RMSE=0.297)
    AB02_Oscar: LG_1p6mps (RMSE=0.214)
    AB03_Ilseung: LG_1p6mps (RMSE=0.203)
    AB05_Maria: LG_0p8mps (R2=0.169, RMSE=0.578), RD_0p8mps (RMSE=0.226)
    AB06_Jimin: LG_0p8mps (RMSE=0.214), RD_0p8mps (R2=0.464, RMSE=0.312)
    AB07_Amy: LG_1p6mps (R2=0.615, RMSE=0.363)
    AB08_Seokhyun: LG_1p2mps (RMSE=0.202), LG_1p6mps (RMSE=0.292), RD_0p8mps (R2=0.556, RMSE=0.353)


,subject,condition,mean_r2,mean_rmse,n,fail_r2,fail_rmse
0,AB01_Jinwoo,LG_0p8mps,0.761816,0.246324,6500,False,True
1,AB01_Jinwoo,LG_1p2mps,0.731526,0.288897,6500,False,True
2,AB01_Jinwoo,LG_1p6mps,0.811261,0.259897,6500,False,True
3,AB01_Jinwoo,RA_0p8mps,0.679239,0.297029,6500,True,True
4,AB02_Oscar,LG_1p6mps,0.884522,0.214143,6500,False,True
5,AB03_Ilseung,LG_1p6mps,0.889715,0.202759,6500,False,True
6,AB05_Maria,LG_0p8mps,0.169326,0.578192,6500,True,True
7,AB05_Maria,RD_0p8mps,0.787473,0.226120,6500,False,True
8,AB06_Jimin,LG_0p8mps,0.738816,0.213743,6500,False,True
9,AB06_Jimin,RD_0p8mps,0.464208,0.311985,6500,True,True


  Note: main cache retains all 40 trials; FILTERED_TRIAL_DATA has 26 (for optional viz only).

Main caches unchanged (all trials): {'hip': '/home/metamobility3/Jinwoo/os_kinetics/analysis/cache/process_awinda_per_joint_hip.npz', 'knee': '/home/metamobility3/Jinwoo/os_kinetics/analysis/cache/process_awinda_per_joint_knee.npz', 'ankle': '/home/metamobility3/Jinwoo/os_kinetics/analysis/cache/process_awinda_per_joint_ankle.npz'}
